In [ ]:
# Improvements :
# Request the amount of Adults and Children for the hotels' search
# Request the amount of Nights for the hotels' search (maximum 4 next days)

In [1]:
import requests
import datetime
import pandas as pd
import csv
from scrapy import Selector

In [2]:
cities = []
with open('cities_with_geoposition.csv', mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        cities.append(row['city']) # 'city' est le nom de ta colonne

print(cities)


['Mont Saint Michel', 'St Malo', 'Bayeux', 'Le Havre', 'Rouen', 'Paris', 'Amiens', 'Lille', 'Strasbourg', 'Chateau du Haut Koenigsbourg', 'Colmar', 'Eguisheim', 'Besancon', 'Dijon', 'Annecy', 'Grenoble', 'Lyon', 'Gorges du Verdon', 'Bormes les Mimosas', 'Cassis', 'Marseille', 'Aix en Provence', 'Avignon', 'Uzes', 'Nimes', 'Aigues Mortes', 'Saintes Maries de la mer', 'Collioure', 'Carcassonne', 'Ariege', 'Toulouse', 'Montauban', 'Biarritz', 'Bayonne', 'La Rochelle']


In [3]:
today = datetime.date.today()
today_str = today.strftime('%Y-%m-%d')

# On crée un écart de 4 jours
delta_days = datetime.timedelta(days=4)

# On l'ajoute à la date d'aujourd'hui
future_date = today + delta_days
future_date_str = future_date.strftime('%Y-%m-%d')
print(f"Aujourd'hui : {today_str}")
print(f"Dans 4 jours : {future_date_str}")

Aujourd'hui : 2026-02-11
Dans 4 jours : 2026-02-15


In [55]:
url_base = "https://www.booking.com/searchresults.fr.html"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'none',
    'Sec-Fetch-User': '?1',
    'Cache-Control': 'max-age=0',
}

list_dict_hotels = []
dict_hotels = {}

for city in cities:
    print(city)
    
    params = {"ss" : f"{city}", "lang" : "fr", "checkin": f"{today_str}", "checkout": f"{future_date_str}"}

    try:
        response = requests.get(url=url_base, params=params, headers=headers )
        print(response.status_code)
        if response.status_code == 200:
            # On donne le HTML brut à Selector
            sel = Selector(text=response.text)

            titres = sel.css('div[data-testid="title"]::text').getall()
            urls = sel.css('a[data-testid="title-link"]::attr(href)').getall()
            scores = sel.xpath('//div[@data-testid="review-score"]/div[@aria-hidden="true"]/text()').getall()

            for t, u, s in zip(titres, urls, scores):
                dict_hotels = {
                    "city": city,
                    "name": t,
                    "url": u,
                    "score": s
                }   
                list_dict_hotels.append(dict_hotels)
            # Check HTML
            #with open("booking_test.html", "w", encoding="utf-8") as f:
                #f.write(response.text)

    except:
        print("Something went wrong during the request")

    break
list_dict_hotels

Mont Saint Michel
200


[{'city': 'Mont Saint Michel',
  'name': 'Ermitage - Mont-Saint-Michel',
  'url': 'https://www.booking.com/hotel/fr/ermitage-mont-saint-michel-beauvoir.fr.html?aid=304142&label=mkt123sc-6d24b79b-f144-4970-adcd-9d047bc80242&ucfs=1&arphpl=1&checkin=2026-02-11&checkout=2026-02-15&group_adults=2&req_adults=2&no_rooms=1&group_children=0&req_children=0&hpos=1&hapos=1&sr_order=popularity&srpvid=ff3f176149774052ab3b9411275bbf8a&srepoch=1770802856&all_sr_blocks=735353103_370791756_2_42_0_136349&highlighted_blocks=735353103_370791756_2_42_0_136349&matching_block_id=735353103_370791756_2_42_0_136349&sr_pri_blocks=735353103_370791756_2_42_0_136349_120416&from=searchresults',
  'score': '8,8'},
 {'city': 'Mont Saint Michel',
  'name': 'Au Mont De La Rive #Jaccuzi et Mont-Saint-Michel #',
  'url': 'https://www.booking.com/hotel/fr/au-mont-de-la-rive-jaccuzi-et-mont-saint-michel.fr.html?aid=304142&label=mkt123sc-6d24b79b-f144-4970-adcd-9d047bc80242&ucfs=1&arphpl=1&checkin=2026-02-11&checkout=2026-02-

In [ ]:
import os
import logging
import scrapy
from scrapy.crawler import CrawlerProcess


class booking_spider(scrapy.Spider):
    # Name of your spider
    name = "booking_spider"

    def start_requests(self):
        list_urls = []
        for city in cities:
            params = {"ss" : f"{city}", "lang" : "fr", "checkin": f"{today_str}", "checkout": f"{future_date_str}"}
            yield scrapy.Request(f"https://www.booking.com/searchresults.en-gb.html?{params}", callback=self.parse, cb_kwargs={'city': city})
    

    # Callback function that will be called when starting your spider
    def parse(self, response, city):
        dict_hotels = {}

        sel = Selector(text=response.text)

        titres = sel.css('div[data-testid="title"]::text').getall()
        urls = sel.css('a[data-testid="title-link"]::attr(href)').getall()
        scores = sel.xpath('//div[@data-testid="review-score"]/div[@aria-hidden="true"]/text()').getall()

        for t, u, s in zip(titres, urls, scores):
            dict_hotels = {
                "city": city,
                "name": t,
                "url": u,
                "score": s
            }   
            yield dict_hotels

        #callback=self.after_search,

# Name of the file where the results will be saved
filename = "list_dict_hotels.json"

# If file already exists, delete it before crawling (because Scrapy will
# concatenate the last and new results otherwise)
if filename in os.listdir("list_from_booking/"):
    os.remove("list_from_booking/" + filename)

# Declare a new CrawlerProcess with some settings
## USER_AGENT => Simulates a browser on an OS
## LOG_LEVEL => Minimal Level of Log
## FEEDS => Where the file will be stored
## More info on built-in settings => https://docs.scrapy.org/en/latest/topics/settings.html?highlight=settings#settings
process = CrawlerProcess(
    settings={
        "USER_AGENT": (
            "Chrome/140.0.0.0"
        ),
        "LOG_LEVEL": logging.INFO,
        "FEEDS": {
            "list_from_booking/" + filename: {"format": "json"},
         
        },
    }
)


# Start the crawling using the spider you defined above
process.crawl(booking_spider)
process.start()


2026-02-11 13:02:48 [scrapy.utils.log] INFO: Scrapy 2.14.1 started (bot: scrapybot)
2026-02-11 13:02:48 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.0.2',
 'libxml2': '2.11.9',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.0',
 'Twisted': '25.5.0',
 'Python': '3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 '
           '64 bit (AMD64)]',
 'pyOpenSSL': '25.3.0 (OpenSSL 3.5.5 27 Jan 2026)',
 'cryptography': '46.0.4',
 'Platform': 'Windows-11-10.0.26200-SP0'}
2026-02-11 13:02:48 [scrapy.addons] INFO: Enabled addons:
[]
2026-02-11 13:02:48 [scrapy.extensions.telnet] INFO: Telnet Password: 769a22da59513e3d
2026-02-11 13:02:49 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.logcount.LogCount',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logstats.LogStats']
2026-02-11 13:02:49 [scrapy.crawler] INFO: Overridden settings:
{'LOG_LEVEL': 2

RuntimeError: This event loop is already running

2026-02-11 13:02:53 [scrapy.core.engine] INFO: Spider opened
2026-02-11 13:02:53 [py.warnings] WARNING: c:\Users\PL06361\python_stats_env\Lib\site-packages\scrapy\core\spidermw.py:490: ScrapyDeprecationWarning: __main__.booking_spider defines the deprecated start_requests() method. start_requests() has been deprecated in favor of a new method, start(), to support asynchronous code execution. start_requests() will stop being called in a future version of Scrapy. If you use Scrapy 2.13 or higher only, replace start_requests() with start(); note that start() is a coroutine (async def). If you need to maintain compatibility with lower Scrapy versions, when overriding start_requests() in a spider class, override start() as well; you can use super() to reuse the inherited start() implementation without copy-pasting. See the release notes of Scrapy 2.13 for details: https://docs.scrapy.org/en/2.13/news.html
  warn(

2026-02-11 13:02:53 [scrapy.extensions.logstats] INFO: Crawled 0 pages (at 0 

: 